In [2]:
# Connect with Google Drive
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Import required libraries
import numpy as np
import pandas as pd

In [4]:
# Set processed data path
processed_data_path = "/content/drive/MyDrive/Credit-Card-Fraud-Detection/processed_data/"

In [5]:
# Load GRU input data
X_train_seq = np.load(processed_data_path + "X_train_seq.npy")
X_val_seq = np.load(processed_data_path + "X_val_seq.npy")
X_test_seq = np.load(processed_data_path + "X_test_seq.npy")

In [6]:
# Load target labels
y_train = pd.read_csv(processed_data_path + "y_train.csv").values.ravel()
y_val = pd.read_csv(processed_data_path + "y_val.csv").values.ravel()
y_test = pd.read_csv(processed_data_path + "y_test.csv").values.ravel()

In [7]:
# Check data shapes
print("X_train_seq shape:", X_train_seq.shape)
print("X_val_seq shape:", X_val_seq.shape)
print("X_test_seq shape:", X_test_seq.shape)

print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

X_train_seq shape: (184421, 30, 1)
X_val_seq shape: (42559, 30, 1)
X_test_seq shape: (56746, 30, 1)
y_train shape: (184421,)
y_val shape: (42559,)
y_test shape: (56746,)


In [8]:
# Check data types
print("X_train_seq dtype:", X_train_seq.dtype)
print("y_train dtype:", y_train.dtype)

# Check missing values
print("Missing values in X_train_seq:", np.isnan(X_train_seq).sum())
print("Missing values in y_train:", np.isnan(y_train).sum())

X_train_seq dtype: float64
y_train dtype: int64
Missing values in X_train_seq: 0
Missing values in y_train: 0


In [9]:
# Check class distribution
print("Training class distribution:")
print(pd.Series(y_train).value_counts())

print("\nValidation class distribution:")
print(pd.Series(y_val).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())

Training class distribution:
0    184114
1       307
Name: count, dtype: int64

Validation class distribution:
0    42488
1       71
Name: count, dtype: int64

Test class distribution:
0    56651
1       95
Name: count, dtype: int64


In [10]:
# Check unique target labels
print("Unique labels:", np.unique(y_train))

Unique labels: [0 1]


In [ ]:
# Install TensorFlow
#!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 777.9 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 77.7 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 63.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


In [15]:
# Import TensorFlow libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [16]:
# Check TensorFlow version
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [19]:
# Import GRU model layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Dense, Dropout

In [20]:
# Build the GRU model
model = Sequential([
    Input(shape=(30, 1)),
    GRU(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

In [21]:
# Display model architecture
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,977 (58.50 KB)

 Trainable params: 14,977 (58.50 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
# Compile the GRU model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [23]:
# Check model configuration
print("Model compiled successfully.")

Model compiled successfully.
